In [1]:
%matplotlib inline
import streamlit as st
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import folium 
import geopandas as gpd 
from streamlit_folium import st_folium
from PIL import Image
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import contextily as ctx
import pandas as pd

2026-05-30 09:55:01.186 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [5]:
cancer_laua = pd.read_csv("data/incidence_laua.csv")
cancer_laua.head()
cancer_laua.info()

<class 'pandas.DataFrame'>
RangeIndex: 810 entries, 0 to 809
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Year                           810 non-null    int64
 1   Gender                         810 non-null    str  
 2   Age at diagnosis               810 non-null    str  
 3   Geography type                 810 non-null    str  
 4   Geography code                 810 non-null    str  
 5   Geography name                 810 non-null    str  
 6   NDRS main                      810 non-null    str  
 7   Count                          810 non-null    int64
 8   Type of rate                   810 non-null    str  
 9   Rate                           810 non-null    str  
 10  95% lower confidence interval  810 non-null    str  
 11  95% upper confidence interval  810 non-null    str  
 12  Flag                           168 non-null    str  
dtypes: int64(2), str(11)
memory usa

In [14]:
cancer_laua['Gender'].value_counts()
cancer_laua['Age at diagnosis'].value_counts()
cancer_laua['NDRS main'].value_counts()
cancer_laua['Geography name '].value_counts()   

Geography name 
Babergh                         18
Basildon                        18
Bedford                         18
Braintree                       18
Breckland                       18
Brentwood                       18
Broadland                       18
Broxbourne                      18
Cambridge                       18
Castle Point                    18
Central Bedfordshire            18
Chelmsford                      18
Colchester                      18
Dacorum                         18
East Cambridgeshire             18
East Hertfordshire              18
East Suffolk                    18
Epping Forest                   18
Fenland                         18
Great Yarmouth                  18
Harlow                          18
Harrow                          18
Hertsmere                       18
Huntingdonshire                 18
Ipswich                         18
King's Lynn and West Norfolk    18
Luton                           18
Maldon                          18
Mid 

In [15]:
cancer_laua['NDRS main'].value_counts()

NDRS main
Bowel          270
Lung           270
Skin cancer    270
Name: count, dtype: int64

In [16]:
cancer_laua.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,NDRS main,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,0,Age-specific,[u],[u],[u],[note2]
1,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,0,Age-specific,[u],[u],[u],[note2]
2,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Skin cancer,0,Age-specific,[u],[u],[u],[note2]
3,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,1,Age-specific,[u],[u],[u],[note2]
4,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,0,Age-specific,[u],[u],[u],[note2]


In [18]:
bowel_cancer = cancer_laua[cancer_laua['NDRS main'] == 'Bowel'] 
bowel_cancer.head(30)

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,NDRS main,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,0,Age-specific,[u],[u],[u],[note2]
3,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,1,Age-specific,[u],[u],[u],[note2]
6,2022,Persons,50 to 59,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,10,Age-specific,70.6,33.8,129.8,NaN
9,2022,Persons,60 to 69,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,17,Age-specific,133,77.4,212.9,NaN
12,2022,Persons,70 to 79,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,23,Age-specific,187.1,118.5,280.7,NaN
15,2022,Persons,80 and over,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,24,Age-specific,340.7,218.2,507,NaN
18,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000066,Basildon,Bowel,0,Age-specific,[u],[u],[u],[note2]
21,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000066,Basildon,Bowel,8,Age-specific,12.6,5.4,24.8,NaN
24,2022,Persons,50 to 59,Local Authority Unitary Authories (LAUA),E07000066,Basildon,Bowel,16,Age-specific,63.7,36.4,103.4,NaN
27,2022,Persons,60 to 69,Local Authority Unitary Authories (LAUA),E07000066,Basildon,Bowel,34,Age-specific,176.2,122,246.2,NaN


In [19]:
# Bowel cancer incidence by local authority in England, 2017-2019
bowel_df = cancer_laua[cancer_laua['NDRS main'] == 'Bowel']
bowel_df.head()


,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,NDRS main,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,0,Age-specific,[u],[u],[u],[note2]
3,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,1,Age-specific,[u],[u],[u],[note2]
6,2022,Persons,50 to 59,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,10,Age-specific,70.6,33.8,129.8,NaN
9,2022,Persons,60 to 69,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,17,Age-specific,133,77.4,212.9,NaN
12,2022,Persons,70 to 79,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Bowel,23,Age-specific,187.1,118.5,280.7,NaN


In [24]:
bowel_2022 = bowel_df.pivot(index=['Geography name ', 'Geography code'], columns='Age at diagnosis', values='Count')
bowel_2022.head()

,Age at diagnosis,00 to 24,25 to 49,50 to 59,60 to 69,70 to 79,80 and over
Geography name,Geography code,,,,,,
Babergh,E07000200,0,1,10,17,23,24
Basildon,E07000066,0,8,16,34,35,33
Bedford,E06000055,0,7,14,24,43,31
Braintree,E07000067,0,7,21,21,32,32
Breckland,E07000143,0,4,18,30,38,39


In [25]:
lung_df = cancer_laua[cancer_laua['NDRS main'] == 'Lung']
lung_df.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,NDRS main,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
1,2022,Persons,00 to 24,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,0,Age-specific,[u],[u],[u],[note2]
4,2022,Persons,25 to 49,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,0,Age-specific,[u],[u],[u],[note2]
7,2022,Persons,50 to 59,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,4,Age-specific,28.2,7.7,72.3,NaN
10,2022,Persons,60 to 69,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,14,Age-specific,109.5,59.8,183.7,NaN
13,2022,Persons,70 to 79,Local Authority Unitary Authories (LAUA),E07000200,Babergh,Lung,38,Age-specific,309.1,218.7,424.2,NaN


In [26]:
skin_df = cancer_laua[cancer_laua['NDRS main'] == 'Skin']
skin_df.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,NDRS main,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag


In [28]:
df_icb = pd.read_csv("data/incidence_icbs.csv")
df_icb.head()

,Year,Gender,Age at diagnosis,Geography type,Geography code,Geography name,Deprivation,Stage,NDRS main,NDRS detailed,Count,Type of rate,Rate,95% lower confidence interval,95% upper confidence interval,Flag
0,2022,Persons,00 to 24,Integrated Care Board,E54000024,"NHS Bedfordshire, Luton and Milton Keynes Inte...",All quintiles,All stages,Bowel,All Bowel,0,Non-standardised,[u],[u],[u],[note2]
1,2022,Persons,00 to 24,Integrated Care Board,E54000024,"NHS Bedfordshire, Luton and Milton Keynes Inte...",All quintiles,All stages,Bowel,Colon,0,Non-standardised,[u],[u],[u],[note2]
2,2022,Persons,00 to 24,Integrated Care Board,E54000024,"NHS Bedfordshire, Luton and Milton Keynes Inte...",All quintiles,All stages,Bowel,Rectosigmoid junction,0,Non-standardised,[u],[u],[u],[note2]
3,2022,Persons,00 to 24,Integrated Care Board,E54000024,"NHS Bedfordshire, Luton and Milton Keynes Inte...",All quintiles,All stages,Bowel,Rectum,0,Non-standardised,[u],[u],[u],[note2]
4,2022,Persons,00 to 24,Integrated Care Board,E54000024,"NHS Bedfordshire, Luton and Milton Keynes Inte...",All quintiles,All stages,Breast,All Breast,1,Non-standardised,[u],[u],[u],[note2]


In [29]:
df_icb['NDRS main'].value_counts()

NDRS main
Bowel          384
Skin cancer    336
Lung           288
Breast          96
Name: count, dtype: int64

In [31]:
df_icb['Geography name '].nunique()

6